# Overnight Variance — where does the risk actually happen?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kris-SF/data-pipelines/blob/main/overnight-variance/overnight_variance.ipynb) &nbsp;&nbsp;&nbsp; [GitHub repo](https://github.com/Kris-SF/data-pipelines/tree/main/overnight-variance)

How much of an asset's daily variance is born **overnight** (while the market is closed) versus **intraday** — and whether a **weekend** carries more risk than a regular weeknight.

Two log returns per day, anchored on the prior close:

- `r_co = ln(open_t / close_{t-1})` — the overnight gap
- `r_cc = ln(close_t / close_{t-1})` — the full close-to-close day

Everything is a **zero-drift** variance (mean of squared returns) and everything is a **ratio**, so drift and annualization cancel.

Edit the `TICKERS` / `START` / `END` config cell and run top to bottom.

In [ ]:
# Bootstrap - runs in both Colab and local Jupyter/VS Code.
# In Colab: clones the repo (so data.py / variance.py import) and installs deps.
# Local: no-op. Run the commented pip line once if you haven't installed requirements.
try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    if not os.path.isdir('data-pipelines'):
        !git clone --quiet --depth 1 https://github.com/Kris-SF/data-pipelines.git
    %cd data-pipelines/overnight-variance
    %pip install --quiet -r requirements.txt

# Local first-time install (uncomment once, then comment back):
# %pip install -r requirements.txt

## 1. Configure and run

`run_overnight_analysis` fetches adjusted OHLC from yfinance and computes the rolling ratio plus the weekend-premium table.

In [ ]:
from variance import run_overnight_analysis, plot_ratio, plot_weekend_premium

TICKERS = ['SPY', 'QQQ', 'USO', 'GLD', 'TLT', 'FXY', 'FXE']

START  = '2023-06-01'
END    = '2026-06-29'
WINDOW = 21            # trailing days for the rolling variance ratio

res = run_overnight_analysis(TICKERS, start=START, end=END, window=WINDOW)
res.pooled             # weekend / regular overnight variance multiple, per ticker

`res` holds everything for further poking:

- `res.pooled` - pooled weekend/regular multiple (+ vol equivalent) per ticker
- `res.premium` - the full half-year bucket table
- `res.ratio` - rolling overnight variance ratio, date x ticker
- `res.returns['r_co']`, `res.returns['r_cc']` - the underlying return panels

## 2. The weekend premium

How much more variance a clean Fri->Mon gap carries than a regular weeknight overnight. A weekend is ~3.5x the *calendar* closed-time of a weeknight, so anything well below 3.5x means overnight risk does **not** scale with how long the market is shut.

In [ ]:
import matplotlib.pyplot as plt
plot_weekend_premium(res)
plt.tight_layout(); plt.show()

## 3. The rolling overnight variance ratio

`Sigma r_co^2 / Sigma r_cc^2` over the trailing window. Above 1.0 the overnight move exceeds the whole day - the session systematically fades the gap. Try just the extremes, e.g. `['SPY', 'USO']`.

In [ ]:
plot_ratio(res, tickers=['SPY', 'USO'])
plt.tight_layout(); plt.show()

## 4. The half-year detail

Per-bucket counts and multiples. The weekend buckets hold only ~20-26 observations each, so trust the `POOLED` rows; the individual half-years are noisy and mostly show *regime dependence* (the premium spikes in stressed periods).

In [ ]:
import pandas as pd
pd.set_option('display.float_format', lambda v: f'{v:.3e}' if abs(v) < 1e-2 else f'{v:.2f}')
res.premium[res.premium.ticker.isin(['SPY', 'USO'])].round(4)

---
### Takeaways

- **The weekend premium is an equity story.** SPY/QQQ run ~1.8x; rates and FX (TLT, FXY, FXE) hug ~1.0x - a weekend night is statistically indistinguishable from a weeknight for them.
- **Overnight risk does not scale with closed time.** Nothing reaches the ~3.5x you'd get if variance accrued with the calendar clock - gap risk is event- and open-driven.
- **Rates/FX move on the *weekday* macro calendar** (CPI, payrolls, FOMC, auctions), so their weekends are quiet; equities carry weekend headline risk.

**Data note:** `auto_adjust=True` back-adjusts for dividends so ex-div drops don't masquerade as overnight gaps (TLT pays monthly). yfinance revises and rate-limits, so exact figures can drift run-to-run; the cross-section is stable.

*If you use options to hedge or invest, check out the [moontower.ai](https://moontower.ai) option trading analytics platform.*